# Libraries + Data Load

In [2]:
# Mounting google drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Data load
data = '/content/drive/My Drive/Capstone Project/Churn Prediction/data/Telco-Customer-Churn.csv'
df = pd.read_csv(data)

# Ek backup copy rakhte hain — original kabhi mat choona!
df_original = df.copy()

print("Data loaded successfully!")
print(f"Shape: {df.shape}")
df.head()

Data loaded successfully!
Shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


# Task 1: TotalCharges Fix
Kya problem hai: TotalCharges str hai, float hona chahiye. 11 rows mein blank string ' ' hai jisko pandas NaN mein convert karega.

In [4]:
# Step 1: String → float (blank strings NaN ban jaayenge automatically)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Step 2: Verify karo
print("TotalCharges dtype ab:", df['TotalCharges'].dtype)
print("NaN count ab:", df['TotalCharges'].isnull().sum())

# Step 3: NaN wali rows dekho
print("\nProblematic rows:")
print(df[df['TotalCharges'].isnull()][['tenure', 'MonthlyCharges', 'TotalCharges']])

TotalCharges dtype ab: float64
NaN count ab: 11

Problematic rows:
      tenure  MonthlyCharges  TotalCharges
488        0           52.55           NaN
753        0           20.25           NaN
936        0           80.85           NaN
1082       0           25.75           NaN
1340       0           56.05           NaN
3331       0           19.85           NaN
3826       0           25.35           NaN
4380       0           20.00           NaN
5218       0           19.70           NaN
6670       0           73.35           NaN
6754       0           61.90           NaN


In [6]:
# Step 4: Drop karo ye 11 rows
df.dropna(subset=['TotalCharges'], inplace=True)

print(f"\nRows after dropping: {df.shape[0]}")  # 7032 hona chahiye


Rows after dropping: 7032


# Task 2: customerID Drop Karo

In [7]:
# customerID = sirf ek unique identifier, koi pattern nahi
df.drop('customerID', axis=1, inplace=True)

print("customerID dropped!")
print(f"Columns remaining: {df.shape[1]}")  # 20 columns

customerID dropped!
Columns remaining: 20


# Task 3: Target Variable Encode Karo

In [8]:
# Churn: 'Yes' → 1, 'No' → 0
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

# Verify
print("Churn value counts:")
print(df['Churn'].value_counts())
print(f"\nChurn rate: {df['Churn'].mean()*100:.2f}%")

Churn value counts:
Churn
0    5163
1    1869
Name: count, dtype: int64

Churn rate: 26.58%


# Task 4: Binary Columns Encode Karo
Ye columns hain jinmein sirf Yes/No hai:

In [9]:
# Pehle dekho kaunse columns Yes/No hain
binary_cols = []
for col in df.columns:
    unique_vals = df[col].unique().tolist()
    if set(unique_vals) == {'Yes', 'No'}:
        binary_cols.append(col)

print("Binary columns found:", binary_cols)

Binary columns found: ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']


In [10]:
# Ek loop mein sab encode karo
for col in binary_cols:
    df[col] = df[col].map({'Yes': 1, 'No': 0})

print("✅ Binary columns encoded!")
print(df[binary_cols].head(3))

✅ Binary columns encoded!
   Partner  Dependents  PhoneService  PaperlessBilling
0        1           0             0                 1
1        0           0             1                 0
2        0           0             1                 1


 # gender aur SeniorCitizen

In [12]:
# gender encode karo
df['gender'] = df['gender'].map({'Male': 1, 'Female': 0})

# SeniorCitizen pehle se hi 0/1 hai — check karo
print("SeniorCitizen unique:", df['SeniorCitizen'].unique())  # [0, 1] ✅

print("✅ gender encoded!")

SeniorCitizen unique: [0 1]
✅ gender encoded!


# Task 5: One-Hot Encoding

In [13]:
# MultipleLines: 'Yes', 'No', 'No phone service'
# InternetService: 'DSL', 'Fiber optic', 'No'
# Contract: 'Month-to-month', 'One year', 'Two year'
# PaymentMethod: 4 types

ohe_cols = ['MultipleLines', 'InternetService', 'Contract', 'PaymentMethod']

print("Before OHE - columns:", df.shape[1])

# pd.get_dummies - drop_first=True to avoid dummy variable trap!
df = pd.get_dummies(df, columns=ohe_cols, drop_first=True)

print("After OHE - columns:", df.shape[1])
print("\nNew columns added:")
for col in df.columns:
    if any(x in col for x in ohe_cols):
        print(" ", col)

Before OHE - columns: 20
After OHE - columns: 25

New columns added:
  MultipleLines_No phone service
  MultipleLines_Yes
  InternetService_Fiber optic
  InternetService_No
  Contract_One year
  Contract_Two year
  PaymentMethod_Credit card (automatic)
  PaymentMethod_Electronic check
  PaymentMethod_Mailed check


# Task 6: Feature Scaling

In [16]:
# Ye 3 columns scale honge
numerical_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

# Pehle train-test split KARO, PHIR scale karo!
# (Lekin woh Step 3 mein hoga — abhi ek important rule yaad karo)
# Abhi ke liye sirf scaler object banaate hain
# Actual fit-transform Step 3 (Model Building) mein hoga

scaler = StandardScaler()
print("✅ Scaler object ready — Step 3 mein use karenge")

✅ Scaler object ready — Step 3 mein use karenge


# Final Cleaned Data Check

In [17]:
print("=" * 50)
print("CLEANING COMPLETE — FINAL SUMMARY")
print("=" * 50)
print(f"\n✅ Shape: {df.shape}")
print(f"✅ Null values: {df.isnull().sum().sum()}")
print(f"✅ All dtypes numeric: {all(df.dtypes != 'object')}")
print(f"\nAll columns:")
for i, col in enumerate(df.columns):
    print(f"  {i+1:2d}. {col} ({df[col].dtype})")

CLEANING COMPLETE — FINAL SUMMARY

✅ Shape: (7032, 25)
✅ Null values: 7032
✅ All dtypes numeric: False

All columns:
   1. gender (float64)
   2. SeniorCitizen (int64)
   3. Partner (int64)
   4. Dependents (int64)
   5. tenure (int64)
   6. PhoneService (int64)
   7. OnlineSecurity (object)
   8. OnlineBackup (object)
   9. DeviceProtection (object)
  10. TechSupport (object)
  11. StreamingTV (object)
  12. StreamingMovies (object)
  13. PaperlessBilling (int64)
  14. MonthlyCharges (float64)
  15. TotalCharges (float64)
  16. Churn (int64)
  17. MultipleLines_No phone service (bool)
  18. MultipleLines_Yes (bool)
  19. InternetService_Fiber optic (bool)
  20. InternetService_No (bool)
  21. Contract_One year (bool)
  22. Contract_Two year (bool)
  23. PaymentMethod_Credit card (automatic) (bool)
  24. PaymentMethod_Electronic check (bool)
  25. PaymentMethod_Mailed check (bool)


# Cleaned Data Save Karo

In [18]:
# Cleaned data save karo taaki agle notebook mein dobara clean na karna pade
df.to_csv('/content/drive/My Drive/Capstone Project/Churn Prediction/data/telco_cleaned.csv', index=False)

print("✅ Cleaned data saved: ../data/telco_cleaned.csv")
print(f"Final shape: {df.shape}")

✅ Cleaned data saved: ../data/telco_cleaned.csv
Final shape: (7032, 25)
